# Duplicated generator events in the checkout files

While checking $\eta\to\gamma\gamma$ decay kinematics (`eta_decay_isotropy_check.ipynb`) two pairs of decays in the Run 5 $\nu$ overlay
file had *bit-identical* rest-frame directions **and** identical $\eta$ 4-momenta, in different runs (25097 and 25117). That is not a
decay-sampling artefact: the whole generator event was duplicated. This notebook quantifies that over **every** checkout file in
`data_files/` (plus the Run 4b re-tuples and the full-oscillation file in `other_files/`): $\nu$, intrinsic $\nu_e$, dirt, NC/CC $\pi^0$,
NuWro, delete-/isotropic-one-gamma, all DetVar variants, and beam-on / beam-off data.

**Duplicate key.** For MC, an event is identified by the exact float32 tuple
(`truth_nuEnergy`, `truth_vtxX`, `truth_vtxY`, `truth_vtxZ`, `truth_nuTime`, `truth_nuPdg`) from `wcpselection/T_eval`, compared
byte-for-byte. Six independent floats cannot collide by chance in $10^7$ events. For data/EXT (no truth) the key is run/subrun/event.
Both keys are also checked *across* files. The scan (`scan_all_files` below, ~1 min with 10 processes) is cached in
`eta_decay_check_cache/duplicate_scan_all_files.npz`; delete the cache to redo it.

> Note: a first version of this notebook used a *scalar* hash (a weighted sum of $E_\nu$, $E_\ell$, daughter count, vertex $z$) which has
> an accidental-collision rate of a few per $10^6$ events. That produced a handful of fake "isolated pairs" (e.g. 10 events in Run 1-3
> `hist_1`, which actually has **zero** duplicates). The byte-exact key used here has none of that.

In [ ]:
import os, glob, re, time, numpy as np, uproot
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from collections import Counter, defaultdict
from multiprocessing import Pool
mpl.rcParams['figure.dpi'] = 110

DATA_DIR  = "/nevis/riverside/data/leehagaman/ngem/data_files"
OTHER_DIR = "/nevis/riverside/data/leehagaman/ngem/other_files"
CACHE_DIR = "eta_decay_check_cache"; os.makedirs(CACHE_DIR, exist_ok=True)
SCAN_CACHE   = os.path.join(CACHE_DIR, "duplicate_scan_all_files.npz")
DETAIL_CACHE = os.path.join(CACHE_DIR, "duplicate_events_run5_detail.npz")
RUN5_FILE = os.path.join(DATA_DIR, "checkout_MCC9.10_Run4acd5_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist_5.root")
RUN_A, RUN_B = 25097, 25117          # the big duplicated block in the Run 5 nu overlay file
TRUTH = ['truth_nuEnergy', 'truth_vtxX', 'truth_vtxY', 'truth_vtxZ', 'truth_nuTime', 'truth_nuPdg']

def short(n):
    for a, b in [('checkout_', ''), ('MCC9.10_', ''), ('_surprise_reco2_hist', ''), ('_surprise_redo_reco2_hist', ''), ('_retuple_retuple_hist', ''),
                 ('_reco2_prod_reco2_hist', ''), ('_v10_04_07_', '_v'), ('.root', ''), ('BNB_', ''), ('_overlay', '')]:
        n = n.replace(a, b)
    return n

def sample_type(n):
    for k, s in [('nuwro', 'NuWro'), ('intrinsic_nue', 'nue'), ('kaon_flux_fix', 'nu'), ('nu_overlay', 'nu'), ('dirt', 'dirt'), ('NCpi0', 'NCpi0'), ('CCpi0', 'CCpi0'),
                 ('delete_one_gamma', 'del1g'), ('isotropic', 'iso1g'), ('numu2nue', 'fullosc'), ('beam_on', 'data'), ('beam_off', 'ext')]:
        if k in n: return s
    return '?'

def run_period(n):
    m = re.search(r'hist_(4a|4b|4c|4d|4bcd|5|1|2|3)(?:_v3|_5e19opendata|_1e19opendata)?\.root$', n)
    if m: return m.group(1)
    if 'Run4b' in n or 'run_4b' in n: return '4b'
    if 'cv_3b' in n: return '3'
    if 'run1' in n: return '1'
    if 'opendata_19550' in n: return '4c'
    return '?'


## 1. Scan every file: duplicates within a file

In [ ]:
def _void(cols):
    arr = np.ascontiguousarray(np.column_stack([np.asarray(c).view(np.uint32) for c in cols]))
    return arr.view(np.dtype((np.void, 4*arr.shape[1]))).ravel()

def scan_one(fn):
    """Per-file: RSE duplicates (all files) and byte-exact truth-key duplicates (MC)."""
    name = os.path.basename(fn)
    try:
        f = uproot.open(fn)
        if 'wcpselection/T_eval;1' not in f.keys(): return name, None
        t = f['wcpselection/T_eval']; is_mc = 'truth_nuEnergy' in t.keys()
        a = t.arrays(['run', 'subrun', 'event'] + (TRUTH if is_mc else []), library='np')
        rse = np.column_stack([a['run'], a['subrun'], a['event']]).astype(np.int32)
        rse_key = _void([rse[:, 0], rse[:, 1], rse[:, 2]])
        u, inv, c = np.unique(rse_key, return_inverse=True, return_counts=True); dup_rse = c[inv] > 1
        res = dict(name=name, n=len(rse), is_mc=is_mc, n_dup_rse=int(dup_rse.sum()), dup_rse_list=rse[dup_rse], rse=rse, rse_key=rse_key)
        if is_mc:
            tk = _void([a[b].astype(np.float32) for b in TRUTH[:5]] + [a['truth_nuPdg'].astype(np.int32)])
            valid = np.isfinite(a['truth_nuEnergy']) & (a['truth_nuEnergy'] > 0)
            u, inv, c = np.unique(tk, return_inverse=True, return_counts=True); dup_t = (c[inv] > 1) & valid
            res.update(truth_key=tk, valid=valid, n_dup_truth=int(dup_t.sum()), dup_truth_rse=rse[dup_t], dup_truth_key=tk[dup_t], n_invalid=int((~valid).sum()))
        return name, res
    except Exception as e:
        print(name, 'FAILED', e); return name, None

def cross_pairs(keys_list, rse_list):
    """Group identical keys across files; return {(file_i, file_j): count} (i<=j) and up to 5 example RSE pairs each."""
    keys = np.concatenate(keys_list); fidx = np.concatenate([np.full(len(k), i, np.int16) for i, k in enumerate(keys_list)]); rses = np.concatenate(rse_list)
    order = np.argsort(keys, kind='stable'); ks = keys[order]
    grp = np.cumsum(~np.concatenate([[False], ks[1:] == ks[:-1]])) - 1
    multi = np.bincount(grp)[grp] > 1; sel = order[multi]; gs = grp[multi]; fs = fidx[sel]; rs = rses[sel]
    counts, examples = defaultdict(int), defaultdict(list)
    bounds = np.flatnonzero(np.concatenate([[True], gs[1:] != gs[:-1], [True]]))
    for b0, b1 in zip(bounds[:-1], bounds[1:]):
        for x in range(b0, b1):
            for y in range(x + 1, b1):
                i, j = sorted((int(fs[x]), int(fs[y]))); counts[(i, j)] += 1
                if len(examples[(i, j)]) < 5: examples[(i, j)].append((rs[x].tolist(), rs[y].tolist()))
    return counts, examples

def scan_all_files():
    files = sorted(glob.glob(DATA_DIR + '/*.root')) + [OTHER_DIR + '/checkout_MCC9.10_Run4b_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist_FSIrwgt.root',
                                                         OTHER_DIR + '/checkout_prodgenie_bnb_overlay_numu2nue_ccactive_run1_PF.root',
                                                         OTHER_DIR + '/run_4b_nu_overlay_splines_kaon_flux_fix.root']
    with Pool(10) as p: results = dict(p.map(scan_one, files, chunksize=1))
    results = {k: v for k, v in results.items() if v is not None}; names = sorted(results); out = {'names': np.array(names)}
    for n in names:
        r = results[n]
        for k in ['n', 'is_mc', 'n_dup_rse', 'dup_rse_list']: out[f'{n}__{k}'] = r[k]
        if r['is_mc']:
            for k in ['n_dup_truth', 'dup_truth_rse', 'n_invalid']: out[f'{n}__{k}'] = r[k]
            out[f'{n}__dup_truth_group'] = np.unique(r['dup_truth_key'], return_inverse=True)[1].astype(np.int32)
    mc = [n for n in names if results[n]['is_mc']]
    counts, ex = cross_pairs([results[n]['truth_key'][results[n]['valid']] for n in mc], [results[n]['rse'][results[n]['valid']] for n in mc])
    out['mc_names'] = np.array(mc); out['pair_i'] = np.array([k[0] for k in counts], np.int32); out['pair_j'] = np.array([k[1] for k in counts], np.int32)
    out['pair_count'] = np.array(list(counts.values()), np.int64); out['pair_examples'] = np.array([ex[k] for k in counts], dtype=object)
    dat = [n for n in names if not results[n]['is_mc']]
    counts, ex = cross_pairs([results[n]['rse_key'] for n in dat], [results[n]['rse'] for n in dat])
    out['data_names'] = np.array(dat); out['dpair_i'] = np.array([k[0] for k in counts], np.int32); out['dpair_j'] = np.array([k[1] for k in counts], np.int32)
    out['dpair_count'] = np.array(list(counts.values()), np.int64); out['dpair_examples'] = np.array([ex[k] for k in counts], dtype=object)
    return out

if not os.path.exists(SCAN_CACHE):
    t0 = time.time(); np.savez(SCAN_CACHE, **scan_all_files()); print(f"scan done in {time.time()-t0:.0f} s")
scan = dict(np.load(SCAN_CACHE, allow_pickle=True))
names = list(scan['names'])
print(len(names), "files scanned")

In [ ]:
rows = []
for n in names:
    is_mc = bool(scan[f'{n}__is_mc']); N = int(scan[f'{n}__n'])
    ndup = int(scan[f'{n}__n_dup_truth']) if is_mc else int(scan[f'{n}__n_dup_rse'])
    rows.append(dict(name=n, type=sample_type(n), period=run_period(n), detvar='DetVar' in n, is_mc=is_mc, n=N, n_dup_rse=int(scan[f'{n}__n_dup_rse']), n_dup=ndup, frac=100*ndup/N))
tab = pd.DataFrame(rows).sort_values(['is_mc', 'type', 'detvar', 'period', 'name'], ascending=[False, True, True, True, True])
pd.set_option('display.width', 250); pd.set_option('display.max_rows', 200)
print(tab.assign(name=tab.name.map(short))[['name', 'type', 'period', 'detvar', 'n', 'n_dup_rse', 'n_dup', 'frac']].to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print(f"\nrun/subrun/event duplicates within any file: {tab.n_dup_rse.sum()}")
mc_tab = tab[tab.is_mc]
print(f"MC events in duplicated truth groups: {mc_tab.n_dup.sum()} of {mc_tab.n.sum()} ({100*mc_tab.n_dup.sum()/mc_tab.n.sum():.3f}%)")
nw = mc_tab.type == 'NuWro'
print(f"  excluding NuWro: {mc_tab[~nw].n_dup.sum()} of {mc_tab[~nw].n.sum()} ({100*mc_tab[~nw].n_dup.sum()/mc_tab[~nw].n.sum():.4f}%)")
print(f"  NuWro only:      {mc_tab[nw].n_dup.sum()} of {mc_tab[nw].n.sum()} ({100*mc_tab[nw].n_dup.sum()/mc_tab[nw].n.sum():.2f}%)")
nom_nu = mc_tab[(mc_tab.type == 'nu') & ~mc_tab.detvar & ~mc_tab.name.str.contains('FSIrwgt|kaon')]
print(f"  nominal nu overlay files: {nom_nu.n_dup.sum()} of {nom_nu.n.sum()} ({100*nom_nu.n_dup.sum()/nom_nu.n.sum():.4f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
t = mc_tab.reset_index(drop=True)
colors = {s: c for s, c in zip(sorted(t.type.unique()), plt.cm.tab10.colors)}
ax.bar(range(len(t)), t.frac.clip(lower=1e-4), color=[colors[s] for s in t.type], edgecolor=['k' if d else 'none' for d in t.detvar])
ax.set_yscale('log'); ax.set_ylabel('% of events in a duplicated truth group'); ax.set_ylim(1e-4, 30)
ax.set_xticks(range(len(t))); ax.set_xticklabels(t.name.map(short), rotation=90, fontsize=6)
for s, c in colors.items(): ax.bar([0], [0], color=c, label=s)
ax.bar([0], [0], color='w', edgecolor='k', label='(black edge = DetVar)'); ax.legend(ncol=5, fontsize=8, loc='upper left')
ax.set_title('Within-file duplicated generator events, all MC checkout files (byte-exact 6-field truth key)'); plt.tight_layout(); plt.show()

Observations:

* **No file has a duplicated run/subrun/event**, MC or data. The duplicates are always re-labelled copies.
* Every GENIE sample type shows a trickle of duplicated events at the $10^{-4}$-$10^{-3}$ level: $\nu$ overlay (Runs 4a-5, but **zero** in Runs 1-3),
  intrinsic $\nu_e$ (Run 5), dirt (4d), NC$\pi^0$ (4d, 5). Each DetVar variant of a given run period carries the *same* count
  (e.g. 306 in every Run 4d variant), as it must, since all variants re-simulate one shared generator sample.
* **NuWro Runs 1-3 is different in kind**: 10.4 % (Run 1), 4.6 % (Run 2) and 1.5 % (Run 3) of events are duplicates, some appearing three
  times. NuWro Runs 4a/4c/5 have none. See section 3.
* Data and EXT files have no duplicated events, within or across files.

## 2. Duplicates across files

In [ ]:
mc = list(scan['mc_names']); M = len(mc)
mat = np.zeros((M, M))
for i, j, c in zip(scan['pair_i'], scan['pair_j'], scan['pair_count']):
    if i != j: mat[i, j] = mat[j, i] = c
order = sorted(range(M), key=lambda k: (sample_type(mc[k]), run_period(mc[k]), 'DetVar' not in mc[k], mc[k]))
fig, ax = plt.subplots(figsize=(13, 12))
sub = mat[np.ix_(order, order)]
im = ax.imshow(np.where(sub > 0, sub, np.nan), norm=mpl.colors.LogNorm(vmin=1, vmax=mat.max()), cmap='viridis')
ax.set_xticks(range(M)); ax.set_yticks(range(M)); ax.set_xticklabels([short(mc[k]) for k in order], rotation=90, fontsize=6); ax.set_yticklabels([short(mc[k]) for k in order], fontsize=6)
plt.colorbar(im, ax=ax, fraction=0.03, label='events with identical truth in both files')
ax.set_title('Cross-file identical generator events (MC). Blocks = DetVar families / re-tuples (expected); off-block dots = repeated generator jobs.', fontsize=10)
plt.tight_layout(); plt.show()

print("Cross-file matches between DIFFERENT sample types or run periods (i.e. not DetVar-family / re-tuple overlap):")
unexpected = []
for k in np.argsort(-scan['pair_count']):
    i, j, c = int(scan['pair_i'][k]), int(scan['pair_j'][k]), int(scan['pair_count'][k])
    if i == j: continue
    fi, fj = (sample_type(mc[i]), run_period(mc[i])), (sample_type(mc[j]), run_period(mc[j]))
    if fi == fj: continue
    unexpected.append((c, mc[i], mc[j], scan['pair_examples'][k][0]))
print(f"{'count':>6s}  {'file A':44s} {'file B':44s} example RSE (A) -> (B)")
for c, a, b, ex in unexpected[:40]:
    print(f"{c:6d}  {short(a):44s} {short(b):44s} {ex[0]} -> {ex[1]}")
print(f"... {len(unexpected)} such file pairs in total (each Run 4d/5 generator duplicate is counted once per DetVar variant)")
print("\nData / EXT: identical run/subrun/event appearing in two different files:", int(scan['dpair_count'].sum()) if len(scan['dpair_count']) else 0)

The large blocks are expected: DetVar variants of one run period share their generator sample with each other (and, for Run 3, ~82 % with the
nominal Run 1-3 `hist_3` $\nu$ overlay); the Run 4b `kaon_flux_fix` file is a re-tuple of the Run 4b $\nu$ overlay.

The off-block entries are the interesting ones: the **same generator event appears in different run periods and even different sample
types** (Run 4d DetVar generator sample vs nominal Run 4c and Run 5 $\nu$ overlay; Run 1-3 `hist_2` vs Run 4d; Run 3 DetVar vs Run 4d
DetVar; Run 5 DetVar vs the delete-one-gamma sample, which is derived from the $\nu$ overlay generator files). A generator job that
produces identical output in two places is a random-seed collision at the generation stage, and evidently it happened at a low rate
throughout the MicroBooNE MCC9 generator production, not just inside one file.

## 3. NuWro Runs 1-3: 1.5-10 % duplicated events

In [ ]:
nuwro = [n for n in names if sample_type(n) == 'NuWro']
fig, axs = plt.subplots(1, 3, figsize=(16, 4.5))
for n in nuwro:
    N = int(scan[f'{n}__n']); rse = scan[f'{n}__dup_truth_rse']; grp = scan[f'{n}__dup_truth_group']
    if len(rse) == 0: print(f"{short(n):40s} {N:7d} events, no duplicates"); continue
    gsize = np.bincount(grp); sizes = Counter(gsize[gsize > 0].tolist())
    groups = defaultdict(list)
    for k, g in enumerate(grp): groups[g].append(k)
    runsets = Counter(tuple(sorted(set(rse[v, 0].tolist()))) for v in groups.values())
    same_run = sum(1 for v in groups.values() if len(set(rse[v, 0].tolist())) == 1)
    print(f"{short(n):40s} {N:7d} events, {len(rse):6d} duplicated ({100*len(rse)/N:5.2f}%), {len(groups)} groups, group sizes {dict(sizes)}, "
          f"{same_run} groups within one run, {len(runsets)} distinct run-sets; most common: {runsets.most_common(3)}")
    if n.endswith('_1.root'):
        pairs_ = [(v[0], v[1]) for v in groups.values() if len(v) >= 2]
        ra = np.array([rse[a, 0] for a, b in pairs_]); rb = np.array([rse[b, 0] for a, b in pairs_])
        lo, hi = np.minimum(ra, rb), np.maximum(ra, rb)
        axs[0].scatter(lo, hi, s=2, alpha=0.3); axs[0].set_xlabel('lower run of a duplicated pair'); axs[0].set_ylabel('higher run'); axs[0].set_title(short(n) + ': which runs share events', fontsize=9)
        cnt = Counter(zip(lo.tolist(), hi.tolist()))
        axs[1].hist(list(cnt.values()), bins=np.arange(0, 200, 5)); axs[1].set_xlabel('duplicated events per (run, run) pair'); axs[1].set_ylabel('run pairs'); axs[1].set_title('blocks of ~150 events = one generator job reused', fontsize=9)
        allrun = uproot.open(os.path.join(DATA_DIR, n))['wcpselection/T_eval'].arrays(['run'], library='np')['run']
        per_run = Counter(allrun.tolist()); dup_run = Counter(rse[:, 0].tolist())
        runs_sorted = sorted(per_run); frac = [100*dup_run.get(r_, 0)/per_run[r_] for r_ in runs_sorted]
        axs[2].plot(runs_sorted, frac, '.', ms=3); axs[2].set_xlabel('run'); axs[2].set_ylabel("% of the run's events that are duplicates"); axs[2].set_title('duplication is spread over the whole run range', fontsize=9)
plt.tight_layout(); plt.show()

The NuWro Run 1 file has ~12 700 duplicated groups, almost all pairs between two *different* runs, in blocks of ~150 events per run pair
(i.e. whole generator jobs reused under a new run number), spread across the full run range. This is the same mechanism as for GENIE,
but at a rate two orders of magnitude higher. For the NuWro fake-data study (`nuwro_fake_data_distributions.ipynb`) it means the Run 1-3
NuWro sample has ~5 % fewer statistically independent events than its row count suggests; the duplicated events have independent
detector simulation and reconstruction, so histograms are not biased, only slightly over-confident in their MC statistical error.

### Per-run view: how much of a run is duplicated?

For every run number that contains at least one duplicated event: what fraction of that run's events are duplicates, how many partner runs
share them, and how many runs are copied *in full*. Uses the MCC9.10 Wire-Cell files (own truth key).

In [ ]:
PER_RUN_FILES = {'NuWro Run 1': 'checkout_MCC9.10_Run123_v10_04_07_23_BNB_nuwro_overlay_surprise_reco2_hist_1.root',
                 'NuWro Run 2': 'checkout_MCC9.10_Run123_v10_04_07_23_BNB_nuwro_overlay_surprise_reco2_hist_2.root',
                 'NuWro Run 3': 'checkout_MCC9.10_Run123_v10_04_07_23_BNB_nuwro_overlay_surprise_reco2_hist_3.root',
                 'nu overlay Run 4a': 'checkout_MCC9.10_Run4acd5_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist_4a.root',
                 'nu overlay Run 4b': 'checkout_MCC9.10_Run4b_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist.root',
                 'nu overlay Run 4c': 'checkout_MCC9.10_Run4acd5_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist_4c.root',
                 'nu overlay Run 4d': 'checkout_MCC9.10_Run4acd5_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist_4d.root',
                 'nu overlay Run 5': 'checkout_MCC9.10_Run4acd5_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist_5.root'}
edges = [0, 0.1, 0.25, 0.5, 0.75, 0.9, 0.9999, 1.0001]
rows_r = []; fracs = {}
for lab, fn_ in PER_RUN_FILES.items():
    a = uproot.open(os.path.join(DATA_DIR, fn_))['wcpselection/T_eval'].arrays(['run'] + TRUTH, library='np')
    tk = _void([a[x].astype(np.float32) for x in TRUTH[:5]] + [a['truth_nuPdg'].astype(np.int32)])
    u, inv, c = np.unique(tk, return_inverse=True, return_counts=True); dup = c[inv] > 1
    runs = a['run']; per_run = Counter(runs.tolist()); dup_run = Counter(runs[dup].tolist())
    groups = defaultdict(list)
    for i in np.where(dup)[0]: groups[inv[i]].append(i)
    partners = defaultdict(set)
    for v in groups.values():
        rs = set(runs[v].tolist())
        for r_ in rs: partners[r_] |= (rs - {r_})
    fr = {r_: dup_run[r_]/per_run[r_] for r_ in dup_run}; fracs[lab] = fr
    n = len(fr); full = sum(1 for f_ in fr.values() if f_ >= 1.0)
    rows_r.append(dict(sample=lab, runs_in_file=len(per_run), runs_with_dups=n, fully_duplicated_runs=full, fully_duplicated_pct=100*full/max(n, 1),
                       runs_ge_50pct=sum(1 for f_ in fr.values() if f_ >= 0.5), runs_with_single_partner=sum(1 for r_ in fr if len(partners[r_]) == 1),
                       median_events_per_run=int(np.median(list(per_run.values()))), median_events_fully_dup_run=int(np.median([per_run[r_] for r_, f_ in fr.items() if f_ >= 1.0])) if full else 0,
                       example_full_runs=[(r_, per_run[r_], sorted(partners[r_])) for r_, f_ in list(fr.items()) if f_ >= 1.0][:3]))
tr = pd.DataFrame(rows_r)
print(tr.drop(columns='example_full_runs').to_string(index=False, float_format=lambda x: f'{x:.1f}'))
print("\nexamples of fully duplicated runs (run, events in run, partner runs):")
for r_ in rows_r:
    if r_['example_full_runs']: print(f"   {r_['sample']}: {r_['example_full_runs']}")

fig, axs = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, lab in zip(axs, ['NuWro Run 1', 'NuWro Run 2', 'NuWro Run 3']):
    ax.hist(list(fracs[lab].values()), bins=edges, edgecolor='k')
    ax.set_xlabel("fraction of the run's events that are duplicates"); ax.set_ylabel('runs'); ax.set_title(f'{lab}: {len(fracs[lab])} runs with duplicates', fontsize=10)
    ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
plt.tight_layout(); plt.show()

Most runs that contain duplicates are only partly affected (typically under a quarter of their events), and a duplicated run usually
shares its events with a single partner run. Runs copied in full are the exception and are small runs: 43 of the 223 affected runs in NuWro
Run 1 (19 %), 13 of 473 in Run 2 (3 %), 4 of 171 in Run 3 (2 %), with 6-44 events each. In the $\nu$ overlay Runs 4-5 no run is copied in
full; the handful of affected runs there have under 10 % of their events duplicated.

### Run/subrun view: where do the copies go?

Same questions one level down, at the (run, subrun) level, which is closer to the generator-job granularity:
are there copies *inside* one subrun, do a subrun's copies go to one partner subrun or many, how many subruns are copied in full,
and how does the number of partners scale with the size of the run or subrun?

In [ ]:
def partner_structure(runs, subs, groups, key_of):
    """For each unit (run or (run,subrun)) with duplicated events: n events duplicated, Counter(partner unit -> n shared events),
    and the number of duplicate groups that lie entirely inside one unit."""
    per_unit = defaultdict(Counter); within = 0; within_examples = []
    for v in groups.values():
        units = [key_of(i) for i in v]
        if len(set(units)) == 1:
            within += 1
            if len(within_examples) < 3: within_examples.append([(int(runs[i]), int(subs[i])) for i in v])
        for i, ui in zip(v, units):
            for j, uj in zip(v, units):
                if i != j and ui != uj: per_unit[ui][uj] += 1
    return per_unit, within, within_examples

rows_s = []; scatter = {}
for lab, fn_ in PER_RUN_FILES.items():
    a = uproot.open(os.path.join(DATA_DIR, fn_))['wcpselection/T_eval'].arrays(['run', 'subrun'] + TRUTH, library='np')
    tk = _void([a[x].astype(np.float32) for x in TRUTH[:5]] + [a['truth_nuPdg'].astype(np.int32)])
    u, inv, c = np.unique(tk, return_inverse=True, return_counts=True); dup = c[inv] > 1
    runs, subs = a['run'], a['subrun']
    groups = defaultdict(list)
    for i in np.where(dup)[0]: groups[inv[i]].append(i)
    if not groups: rows_s.append(dict(sample=lab, subruns_in_file=len(set(zip(runs.tolist(), subs.tolist()))), subruns_with_dups=0)); continue
    size_run = Counter(runs.tolist()); size_sub = Counter(zip(runs.tolist(), subs.tolist()))
    dup_sub = Counter(zip(runs[dup].tolist(), subs[dup].tolist()))
    p_sub, within_sub, ex_sub = partner_structure(runs, subs, groups, lambda i: (int(runs[i]), int(subs[i])))
    p_run, within_run, _ = partner_structure(runs, subs, groups, lambda i: int(runs[i]))
    frac_sub = {s_: dup_sub[s_]/size_sub[s_] for s_ in dup_sub}
    npart_sub = np.array([len(p_sub[s_]) for s_ in p_sub]); top_sub = np.array([max(pc.values())/sum(pc.values()) for pc in p_sub.values()])
    rows_s.append(dict(sample=lab, subruns_in_file=len(size_sub), subruns_with_dups=len(dup_sub), groups_inside_one_subrun=within_sub, groups_inside_one_run=within_run,
                       fully_copied_subruns=sum(1 for f_ in frac_sub.values() if f_ >= 1.0), median_frac_dup=float(np.median(list(frac_sub.values()))),
                       one_partner_subrun_pct=100*np.mean(npart_sub == 1), ge90_to_top_partner_pct=100*np.mean(top_sub >= 0.9), max_partners=int(npart_sub.max()),
                       median_subrun_size=int(np.median(list(size_sub.values()))), median_size_dup_subrun=int(np.median([size_sub[s_] for s_ in dup_sub]))))
    scatter[lab] = dict(sub_size=np.array([size_sub[s_] for s_ in p_sub]), sub_np=npart_sub, sub_frac=np.array([frac_sub[s_] for s_ in p_sub]),
                        run_size=np.array([size_run[r_] for r_ in p_run]), run_np=np.array([len(p_run[r_]) for r_ in p_run]),
                        all_sub_size=np.array(list(size_sub.values())), all_run_size=np.array(list(size_run.values())))
    if within_sub: print(f"{lab}: groups inside one subrun, examples:", ex_sub)
ts = pd.DataFrame(rows_s)
print(ts.to_string(index=False, float_format=lambda x: f'{x:.2f}'))

In [ ]:
labs_plot = [l_ for l_ in ['NuWro Run 1', 'NuWro Run 2', 'NuWro Run 3'] if l_ in scatter]
fig, axs = plt.subplots(2, len(labs_plot), figsize=(5.4*len(labs_plot), 8.5), constrained_layout=True)
rng = np.random.default_rng(0)
for k_, lab in enumerate(labs_plot):
    s = scatter[lab]
    ax = axs[0, k_]
    sc = ax.scatter(s['sub_size'], s['sub_np'] + rng.uniform(-0.15, 0.15, len(s['sub_np'])), s=8, alpha=0.5, c=s['sub_frac'], cmap='viridis', vmin=0, vmax=1)
    ax.set_xscale('log'); ax.set_xlabel('events in the (run, subrun)'); ax.set_ylabel('number of partner subruns'); ax.set_title(f'{lab}: subrun level ({len(s["sub_np"])} subruns with duplicates)', fontsize=9)
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
    ax.axvline(np.median(s['all_sub_size']), color='gray', ls=':', lw=1, label='median subrun size (all subruns)'); ax.legend(fontsize=7, loc='upper left')
    ax = axs[1, k_]
    ax.scatter(s['run_size'], s['run_np'] + rng.uniform(-0.15, 0.15, len(s['run_np'])), s=12, alpha=0.6)
    ax.set_xscale('log'); ax.set_xlabel('events in the run'); ax.set_ylabel('number of partner runs'); ax.set_title(f'{lab}: run level ({len(s["run_np"])} runs with duplicates)', fontsize=9)
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
    ax.axvline(np.median(s['all_run_size']), color='gray', ls=':', lw=1, label='median run size (all runs)'); ax.legend(fontsize=7, loc='upper left')
fig.colorbar(sc, ax=axs[0, :].tolist(), shrink=0.9, pad=0.01, label="fraction of the subrun's events duplicated")
plt.show()

**Within a subrun: never.** No duplicate group has both copies in the same (run, subrun), in any file. Copies inside the same *run* exist
only in NuWro Run 1 (596 groups), always in two different subruns of one of the very large runs, e.g. subrun 32 of run 7049 reappearing as
subrun 1685 of run 7049.

**A copied subrun is usually copied whole and lands in one place.** After the event filtering, a subrun in these files holds only ~5-10
events (it is a slice of one generator job). Of the subruns that contain duplicates, 85 % (Run 1), 77 % (Run 2) and 81 % (Run 3) are
duplicated in full, and their copies always sit in a *single* partner run (Runs 2 and 3: 100 % of affected subruns; Run 1: 90 %). When a
subrun has two or three partner subruns, they are neighbouring subruns of that one partner run: the copied job was cut into subruns at
slightly different event boundaries on the two sides, so a block of ~50 events straddles a subrun edge. It is never "a few events from run A,
a few from run B".

**Number of partners scales with size, and the outliers are the mega-runs.** At the subrun level the partner count is 1-3 for ordinary
subruns. NuWro Run 1 also contains subruns of ~50 events (whole jobs kept intact) with up to 18 partner subruns: these belong to the handful of
runs with 10⁴ events (7003, 7004, 7010, 7017, 7020, 7021, 7049, 7054), which were evidently assembled from hundreds of jobs, a good fraction of
them copies of jobs also used elsewhere. At the run level (bottom row) the partner count is 1 for the typical run and rises steeply only for
those mega-runs, reaching ~40 partner runs for the largest. In Runs 2 and 3, where run sizes are uniform (~100 events), no run has more than
five partners and most have one.

### POT cost of removing the duplicates

`wcpselection/T_pot` gives the POT per (run, subrun). In the NuWro files it is exactly proportional to the number of events in the subrun
(1.2×10¹⁵ POT per event, constant to 5 digits), i.e. the POT was assigned per generated event. Two consequences:

* a duplicated event carries its own POT, so the duplicates do **not** bias the events-per-POT normalisation of the sample; they only reduce
  the number of statistically independent events;
* removing a duplicate together with its POT is self-consistent, and any removal scheme costs exactly (events removed) × 1.2×10¹⁵ POT.

Three schemes are compared: (a) drop every subrun that contains a duplicated event; (b) the minimum-POT set of subruns whose removal
leaves no duplicate pair, solved exactly as a weighted vertex cover (one binary per subrun, one constraint per pair of subruns that share an
event, HiGHS via `scipy.optimize.milp`); (c) the floor, dropping one copy of each duplicated event individually. The (run, subrun) lists for
scheme (b) are written to `eta_decay_check_cache/nuwro_duplicate_subruns_to_remove_run{1,2,3}.csv`.

In [ ]:
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import lil_matrix

rows_pot = []
for lab, fn_ in PER_RUN_FILES.items():
    f_ = uproot.open(os.path.join(DATA_DIR, fn_))
    p_ = f_['wcpselection/T_pot'].arrays(['runNo', 'subRunNo', 'pot_tor875good'], library='np')
    pot = {(int(r_), int(s_)): float(v_) for r_, s_, v_ in zip(p_['runNo'], p_['subRunNo'], p_['pot_tor875good'])}; total = sum(pot.values())
    a = f_['wcpselection/T_eval'].arrays(['run', 'subrun'] + TRUTH, library='np')
    nev = Counter(zip(a['run'].tolist(), a['subrun'].tolist()))
    ratio = np.array([pot[k_]/nev[k_] for k_ in nev]); per_event = ratio.max()/ratio.min() < 1.001
    tk = _void([a[x].astype(np.float32) for x in TRUTH[:5]] + [a['truth_nuPdg'].astype(np.int32)])
    u, inv, c = np.unique(tk, return_inverse=True, return_counts=True); dup = c[inv] > 1
    groups = defaultdict(list)
    for i in np.where(dup)[0]: groups[inv[i]].append(i)
    if not groups: continue
    sub_of = lambda i: (int(a['run'][i]), int(a['subrun'][i]))
    dsubs = sorted({sub_of(i) for i in np.where(dup)[0]}); idx = {s_: k_ for k_, s_ in enumerate(dsubs)}
    edges = set()
    for v in groups.values():
        ss = sorted({sub_of(i) for i in v})
        for x_ in range(len(ss)):
            for y_ in range(x_ + 1, len(ss)): edges.add((idx[ss[x_]], idx[ss[y_]]))
    edges = sorted(edges); n = len(dsubs)
    A = lil_matrix((len(edges), n))
    for e_, (i, j) in enumerate(edges): A[e_, i] = 1; A[e_, j] = 1
    w = np.array([pot[s_] for s_ in dsubs])
    res = milp(c=w, constraints=LinearConstraint(A.tocsr(), lb=np.ones(len(edges)), ub=np.full(len(edges), np.inf)), integrality=np.ones(n), bounds=Bounds(0, 1))
    assert res.status == 0, res.message
    remove = np.round(res.x).astype(bool)
    naive = w.sum(); optimal = float(w[remove].sum())
    floor = sum(len(v) - 1 for v in groups.values()) * ratio.mean() if per_event else np.nan
    frac_dup = np.array([sum(1 for i in np.where(dup)[0] if sub_of(i) == s_)/nev[s_] for s_ in dsubs])
    rows_pot.append(dict(sample=lab, total_POT=total, POT_per_event_constant=per_event, subruns_with_dups=n, subrun_pairs=len(edges),
                         a_drop_all_pct=100*naive/total, b_optimal_pct=100*optimal/total, b_subruns_removed=int(remove.sum()),
                         b_removed_fully_dup_pct=100*np.mean(frac_dup[remove] >= 1), c_floor_pct=100*floor/total))
    if lab.startswith('NuWro'):
        out_csv = os.path.join(CACHE_DIR, f"nuwro_duplicate_subruns_to_remove_run{lab[-1]}.csv")
        pd.DataFrame([dict(run=s_[0], subrun=s_[1], pot=pot[s_], n_events=nev[s_]) for s_, r_ in zip(dsubs, remove) if r_]).sort_values(['run', 'subrun']).to_csv(out_csv, index=False)
tpot = pd.DataFrame(rows_pot)
print(tpot.to_string(index=False, float_format=lambda x: f'{x:.3g}' if abs(x) > 1e6 else f'{x:.2f}'))

Dropping every subrun that contains a duplicate costs 12.0 % / 5.3 % / 1.7 % of the POT in NuWro Runs 1 / 2 / 3. The optimal subrun
selection costs 5.3 % / 2.3 % / 0.75 %, exactly the floor of removing one copy per duplicated event. It achieves that because the optimum
always removes a *fully* duplicated subrun of each pair and keeps its partner (which may also carry non-duplicated events), so nothing is
lost beyond the copies themselves. In practice the recipe is: for each duplicate pair, drop the member whose subrun is 100 % duplicated;
that is what the CSV lists contain.

For the $\nu$ overlay files the POT per subrun is not per-event (jobs of differing size), the numbers are at the 10⁻⁴ level, and no
removal is warranted.

## 3b. Is the NuWro duplication already in the older MCC9 production?

The MCC9 (pre-MCC9.10) NuWro fake-data checkouts in `other_files/nuwro_mcc9_files/` have **all truth stripped** (`mc_isnu = 0`,
`truth_Ntrack = 0`, no `truth_*` in `T_eval`), so the truth key cannot be built from them directly. They are, however, the same
generator events as the MCC9.10 NuWro files: 94-98 % of their run/subrun/event labels are in common and the run ranges coincide. So the
truth key of each MCC9 event is taken from the MCC9.10 event with the same run/subrun/event, and the duplicate search is repeated inside
each MCC9 file. If the duplication were introduced by the MCC9.10 re-processing, it would not show up here.

In [ ]:
MCC9_DIR = os.path.join(OTHER_DIR, "nuwro_mcc9_files")
MCC9_CACHE = os.path.join(CACHE_DIR, "duplicate_scan_nuwro_mcc9.npz")
MCC9_PAIRS = [(1, "checkout_fakedata_nuwro_run1.root", "checkout_MCC9.10_Run123_v10_04_07_23_BNB_nuwro_overlay_surprise_reco2_hist_1.root"),
              (2, "checkout_fakedata_nuwro_run2.root", "checkout_MCC9.10_Run123_v10_04_07_23_BNB_nuwro_overlay_surprise_reco2_hist_2.root"),
              (3, "checkout_fakedata_nuwro_run3.root", "checkout_MCC9.10_Run123_v10_04_07_23_BNB_nuwro_overlay_surprise_reco2_hist_3.root")]

def scan_nuwro_mcc9():
    out = {}
    for run, old, new in MCC9_PAIRS:
        fo = uproot.open(os.path.join(MCC9_DIR, old))
        pf = fo['wcpselection/T_PFeval'].arrays(['mc_isnu', 'truth_Ntrack'], library='np')
        e = fo['wcpselection/T_eval'].arrays(['run', 'subrun', 'event', 'match_energy', 'match_charge', 'flash_time'], library='np')
        rse_o = _void([e['run'].astype(np.int32), e['subrun'].astype(np.int32), e['event'].astype(np.int32)])
        b = uproot.open(os.path.join(DATA_DIR, new))['wcpselection/T_eval'].arrays(['run', 'subrun', 'event'] + TRUTH, library='np')
        rse_n = _void([b['run'].astype(np.int32), b['subrun'].astype(np.int32), b['event'].astype(np.int32)])
        tk = _void([b[x].astype(np.float32) for x in TRUTH[:5]] + [b['truth_nuPdg'].astype(np.int32)])
        lookup = {r_.tobytes(): t_.tobytes() for r_, t_ in zip(rse_n, tk)}
        keys = [lookup.get(r_.tobytes()) for r_ in rse_o]; has = np.array([k_ is not None for k_ in keys])
        u, inv, c = np.unique(np.array([k_ for k_ in keys if k_ is not None], dtype=object), return_inverse=True, return_counts=True); dup = c[inv] > 1
        u2, inv2, c2 = np.unique(tk, return_inverse=True, return_counts=True)
        # reco-identical events (identical matched-cluster energy, charge and flash time): reused cosmic-overlay data, not generator duplicates
        rk = _void([e['match_energy'].astype(np.float32), e['match_charge'].astype(np.float32), e['flash_time'].astype(np.float32)])
        u3, inv3, c3 = np.unique(rk, return_inverse=True, return_counts=True); rdup = (c3[inv3] > 1) & (e['match_energy'] > 0)
        out[f'run{run}__n'] = len(rse_o); out[f'run{run}__truth_stripped'] = bool(np.all(pf['mc_isnu'] == 0) and np.all(pf['truth_Ntrack'] == 0))
        out[f'run{run}__n_with_key'] = int(has.sum()); out[f'run{run}__n_dup'] = int(dup.sum()); out[f'run{run}__group_sizes'] = np.array(sorted(Counter(c[c > 1].tolist()).items()))
        out[f'run{run}__dup_rse'] = np.column_stack([e['run'][has][dup], e['subrun'][has][dup], e['event'][has][dup]]).astype(np.int32)
        out[f'run{run}__n_reco_dup'] = int(rdup.sum()); out[f'run{run}__mcc910_n'] = len(tk); out[f'run{run}__mcc910_n_dup'] = int((c2[inv2] > 1).sum())
        out[f'run{run}__mcc910_in_mcc9'] = int(np.isin(rse_n, rse_o).sum())
    return out

if not os.path.exists(MCC9_CACHE) or 'run1__truth_stripped' not in np.load(MCC9_CACHE, allow_pickle=True).files:
    np.savez(MCC9_CACHE, **scan_nuwro_mcc9())
m9 = dict(np.load(MCC9_CACHE, allow_pickle=True))

rows9 = []
for run, old, new in MCC9_PAIRS:
    g = lambda k: m9[f'run{run}__{k}']
    rows9.append(dict(run=run, mcc9_file=old, mcc9_events=int(g('n')), truth_stripped=bool(g('truth_stripped')), rse_in_mcc910=int(g('n_with_key')),
                      rse_overlap_pct=100*int(g('n_with_key'))/int(g('n')), mcc9_dup=int(g('n_dup')), mcc9_dup_pct=100*int(g('n_dup'))/int(g('n_with_key')),
                      group_sizes=dict(g('group_sizes').tolist()), mcc910_events=int(g('mcc910_n')), mcc910_dup=int(g('mcc910_n_dup')), mcc910_dup_pct=100*int(g('mcc910_n_dup'))/int(g('mcc910_n')),
                      reco_identical=int(g('n_reco_dup'))))
t9 = pd.DataFrame(rows9)
print(t9.to_string(index=False, float_format=lambda x: f'{x:.2f}'))

fig, ax = plt.subplots(figsize=(7, 3.8))
xx = np.arange(3); w = 0.38
ax.bar(xx - w/2, t9.mcc9_dup_pct, w, label='MCC9 fake-data files (truth key via RSE match)')
ax.bar(xx + w/2, t9.mcc910_dup_pct, w, label='MCC9.10 nuwro_overlay files (own truth key)')
ax.set_xticks(xx); ax.set_xticklabels([f'Run {r}' for r in t9.run]); ax.set_ylabel('% of events in a duplicated generator group'); ax.legend(fontsize=8)
ax.set_title('NuWro duplication: older MCC9 production vs MCC9.10', fontsize=10); plt.tight_layout(); plt.show()

Same fractions in both productions (9.5 / 4.3 / 1.5 % in MCC9, marginally lower only because the ~5 % of MCC9 events without a
run/subrun/event match in MCC9.10 cannot be tested): **the duplication is in the underlying NuWro generator sample**, and predates the
MCC9.10 re-processing.

The `reco_identical` column is a different, benign effect: a few hundred events per file have a bit-identical matched cluster (energy, charge,
flash time) in two different runs, but *different* neutrinos (in MCC9.10 Run 1, 920 such events, none with the same true $E_\nu$, 80 %
with zero match completeness). Those are neutrino events whose Wire-Cell match landed on a cosmic from the overlay data, and the same
beam-off data event was recycled for two MC events. That is normal overlay-sample reuse, not generator duplication.

## 3c. Direct confirmation with the MCC9 PeLEE (Pandora) NuWro ntuples

The MCC9 NuWro samples also exist as PeLEE ntuples (`other_files/nuwro_mcc9_files/pandora/`, tree `nuselection/NeutrinoSelectionFilter`),
which **do** keep the generator truth: `nu_e`, `true_nu_vtx_{x,y,z,t}`, `nu_pdg`. That allows the same byte-exact 6-field key as for the
Wire-Cell files, built entirely from the older production (Runs 1-3, 4a, 4c, 5). Two things need care:

* the PeLEE trees contain some entries with a *repeated run/subrun/event* (several thousand in the Run 5 file, a few hundred in Runs 1 and 3).
  Those are ntuple-level repeats of one and the same event, not generator duplicates, so entries are first de-duplicated by run/subrun/event;
* each PeLEE file is also matched by run/subrun/event to the corresponding MCC9.10 Wire-Cell file to check that the two productions flag the
  same events.

In [ ]:
PANDORA_DIR = os.path.join(OTHER_DIR, "nuwro_mcc9_files", "pandora")
PANDORA_CACHE = os.path.join(CACHE_DIR, "duplicate_scan_nuwro_pandora.npz")
PANDORA_PAIRS = [('Run 1',  'high_stat_prodgenie_bnb_nu_overlay_DetVar_Run1_NuWro_reco2_reco2.root', 'checkout_MCC9.10_Run123_v10_04_07_23_BNB_nuwro_overlay_surprise_reco2_hist_1.root'),
                 ('Run 2',  'high_stat_prodgenie_bnb_nu_overlay_DetVar_Run2_NuWro_reco2_reco2.root', 'checkout_MCC9.10_Run123_v10_04_07_23_BNB_nuwro_overlay_surprise_reco2_hist_2.root'),
                 ('Run 3',  'high_stat_prodgenie_bnb_nu_overlay_DetVar_Run3_NuWro_reco2_reco2.root', 'checkout_MCC9.10_Run123_v10_04_07_23_BNB_nuwro_overlay_surprise_reco2_hist_3.root'),
                 ('Run 4a', 'high_stat_prodgenie_bnb_nu_nuwro_overlay_run4a_pelee.root',             'checkout_MCC9.10_Run45_v10_04_07_23_BNB_nuwro_overlay_surprise_reco2_hist_4a.root'),
                 ('Run 4c', 'high_stat_prodgenie_bnb_nu_nuwro_overlay_run4_pelee.root',              'checkout_MCC9.10_Run45_v10_04_07_23_BNB_nuwro_overlay_surprise_reco2_hist_4c.root'),
                 ('Run 5',  'high_stat_prodgenie_bnb_nu_nuwro_overlay_run5_pelee.root',              'checkout_MCC9.10_Run45_v10_04_07_23_BNB_nuwro_overlay_surprise_reco2_hist_5.root')]
PANDORA_TRUTH = ['nu_e', 'true_nu_vtx_x', 'true_nu_vtx_y', 'true_nu_vtx_z', 'true_nu_vtx_t', 'nu_pdg']

def scan_nuwro_pandora():
    out = {}
    for lab, pf, wf in PANDORA_PAIRS:
        a = uproot.open(os.path.join(PANDORA_DIR, pf))['nuselection/NeutrinoSelectionFilter'].arrays(['run', 'sub', 'evt'] + PANDORA_TRUTH, library='np')
        rse = np.column_stack([a['run'], a['sub'], a['evt']]).astype(np.int32); rk = _void([rse[:, 0], rse[:, 1], rse[:, 2]])
        first = np.unique(rk, return_index=True)[1]                      # one entry per run/subrun/event
        n_rse_dup = len(rk) - len(first)
        tk = _void([a[x].astype(np.float32) for x in PANDORA_TRUTH[:5]] + [a['nu_pdg'].astype(np.int32)])[first]; rse = rse[first]; rk = rk[first]
        valid = np.isfinite(a['nu_e'][first]) & (a['nu_e'][first] > 0)
        u, inv, c = np.unique(tk, return_inverse=True, return_counts=True); dup = (c[inv] > 1) & valid
        groups = defaultdict(list)
        for i in np.where(dup)[0]: groups[inv[i]].append(i)
        sizes = Counter(len(v) for v in groups.values())
        runsets = Counter(tuple(sorted(set(rse[v, 0].tolist()))) for v in groups.values())
        # cross-check against the MCC9.10 Wire-Cell file, event by event
        b = uproot.open(os.path.join(DATA_DIR, wf))['wcpselection/T_eval'].arrays(['run', 'subrun', 'event'] + TRUTH, library='np')
        rkb = _void([b['run'].astype(np.int32), b['subrun'].astype(np.int32), b['event'].astype(np.int32)])
        tkb = _void([b[x].astype(np.float32) for x in TRUTH[:5]] + [b['truth_nuPdg'].astype(np.int32)])
        u2, inv2, c2 = np.unique(tkb, return_inverse=True, return_counts=True); dupb = c2[inv2] > 1
        flag_b = {r_.tobytes(): d_ for r_, d_ in zip(rkb, dupb)}
        has = np.array([r_.tobytes() in flag_b for r_ in rk]); fb = np.array([flag_b.get(r_.tobytes(), False) for r_ in rk])
        out[f'{lab}__n_entries'] = len(a['run']); out[f'{lab}__n_rse_dup_entries'] = int(n_rse_dup); out[f'{lab}__n'] = len(rse)
        out[f'{lab}__n_dup'] = int(dup.sum()); out[f'{lab}__dup_rse'] = rse[dup]; out[f'{lab}__dup_group'] = inv[dup].astype(np.int32)
        out[f'{lab}__group_sizes'] = np.array(sorted(sizes.items())); out[f'{lab}__n_runsets'] = len(runsets); out[f'{lab}__top_runsets'] = np.array(runsets.most_common(3), dtype=object)
        out[f'{lab}__n_in_wc'] = int(has.sum()); out[f'{lab}__agree'] = float(np.mean(dup[has] == fb[has])) if has.sum() else np.nan
        out[f'{lab}__wc_n'] = len(tkb); out[f'{lab}__wc_n_dup'] = int(dupb.sum())
    out['labels'] = np.array([p[0] for p in PANDORA_PAIRS])
    return out

if not os.path.exists(PANDORA_CACHE) or 'Run 1__n_rse_dup_entries' not in np.load(PANDORA_CACHE, allow_pickle=True).files:
    np.savez(PANDORA_CACHE, **scan_nuwro_pandora())
pdz = dict(np.load(PANDORA_CACHE, allow_pickle=True))

rows_p = []
for lab, pf, wf in PANDORA_PAIRS:
    g = lambda k: pdz[f'{lab}__{k}']
    rows_p.append(dict(run=lab, pelee_entries=int(g('n_entries')), repeated_rse_entries=int(g('n_rse_dup_entries')), unique_events=int(g('n')),
                       gen_dup=int(g('n_dup')), gen_dup_pct=100*int(g('n_dup'))/int(g('n')), group_sizes=dict(g('group_sizes').tolist()) if len(g('group_sizes')) else {},
                       run_pairs=int(g('n_runsets')), rse_in_wc_pct=100*int(g('n_in_wc'))/int(g('n')), flag_agreement_pct=100*float(g('agree')),
                       wc_mcc910_dup_pct=100*int(g('wc_n_dup'))/int(g('wc_n'))))
tp = pd.DataFrame(rows_p)
print(tp.to_string(index=False, float_format=lambda x: f'{x:.2f}'))

fig, ax = plt.subplots(figsize=(8, 3.8))
xx = np.arange(len(tp)); w = 0.27
ax.bar(xx - w, tp.gen_dup_pct, w, label='MCC9 PeLEE ntuples (own truth key)')
old_wc = {f'Run {r}': 100*int(m9[f'run{r}__n_dup'])/int(m9[f'run{r}__n_with_key']) for r in (1, 2, 3)}
ax.bar(xx, [old_wc.get(l_, 0) for l_ in tp.run], w, label='MCC9 WC fake-data files (key via RSE match, Runs 1-3 only)')
ax.bar(xx + w, tp.wc_mcc910_dup_pct, w, label='MCC9.10 WC nuwro_overlay files (own truth key)')
ax.set_xticks(xx); ax.set_xticklabels(tp.run); ax.set_ylabel('% of events in a duplicated generator group'); ax.legend(fontsize=8)
ax.set_title('NuWro generator duplication seen from three ntuple productions', fontsize=10); plt.tight_layout(); plt.show()

With truth taken directly from the older production, the PeLEE ntuples give 10.6 % / 4.6 % / 1.4 % duplicated generator events in
Runs 1 / 2 / 3 (the Run 3 PeLEE file holds 9 % fewer events than the Wire-Cell one, so more partners are missing there) and none in Runs 4a, 4c and 5, and they flag the *same events* as the MCC9.10 Wire-Cell files (98-100 % agreement on
run/subrun/event-matched entries; the residual is events whose partner is missing from one of the two files). Nothing is duplicated across
the six PeLEE files. This closes the question: the repeated events are a property of the Run 1-3 NuWro generator sample itself, present
identically in every ntuple production made from it.

(The repeated-run/subrun/event entries in the PeLEE Run 5 file, 2496 of them, are a separate ntuple quirk: the same event written twice,
with identical truth, within the same run. They were removed before the search and are not generator duplicates.)

## 4. Structure of one block: runs 25097 and 25117 in the Run 5 $\nu$ overlay file

In [ ]:
def extract_detail(path, run_a, run_b):
    f = uproot.open(path); out = {}
    vt = f["singlephotonana/vertex_tree"]
    vbr = ['run_number','subrun_number','event_number','mctruth_nu_E','mctruth_lepton_E','mctruth_num_daughter_particles',
           'mctruth_nu_vertex_x','mctruth_nu_vertex_y','mctruth_nu_vertex_z','mctruth_nu_pdg','mctruth_interaction_type','mctruth_mode',
           'mctruth_daughters_pdg','mctruth_daughters_E','mctruth_daughters_status_code',
           'reco_vertex_x','reco_vertex_y','reco_vertex_z','reco_asso_showers','reco_asso_tracks']
    a = vt.arrays([b for b in vbr if b in vt.keys()], cut=f'(run_number=={run_a})|(run_number=={run_b})', library='ak')
    for b in a.fields:
        out['vt_'+b] = np.asarray(a[b]) if a[b].ndim == 1 else np.array([np.asarray(v) for v in a[b]], dtype=object)
    te = f["wcpselection/T_eval"]
    ebr = ['run','subrun','event','truth_nuEnergy','truth_vtxX','truth_vtxY','truth_vtxZ','truth_nuTime','truth_nuPdg','truth_isCC','match_isFC',
           'match_completeness_energy','match_energy','truth_energyInside','weight_cv','weight_spline']
    e = te.arrays([b for b in ebr if b in te.keys()], cut=f'(run=={run_a})|(run=={run_b})', library='np')
    for b in e: out['te_'+b] = e[b]
    tb = f["wcpselection/T_BDTvars"]
    bd = tb.arrays(['run','subrun','event','numu_cc_flag','nue_score','numu_score'], cut=f'(run=={run_a})|(run=={run_b})', library='np')
    for b in bd: out['tb_'+b] = bd[b]
    return out

if not os.path.exists(DETAIL_CACHE) or 'te_truth_nuTime' not in np.load(DETAIL_CACHE, allow_pickle=True).files:
    np.savez(DETAIL_CACHE, **extract_detail(RUN5_FILE, RUN_A, RUN_B))
x = dict(np.load(DETAIL_CACHE, allow_pickle=True))
r = x['vt_run_number']
assert np.all((x['te_run'] == r) & (x['te_subrun'] == x['vt_subrun_number']) & (x['te_event'] == x['vt_event_number'])), "tree misalignment"
key = _void([x['te_'+b].astype(np.float32) for b in TRUTH[:5]] + [x['te_truth_nuPdg'].astype(np.int32)])
ia = {k.tobytes(): i for i, k in zip(np.where(r == RUN_A)[0], key[r == RUN_A])}
pairs = [(ia[k.tobytes()], j) for j, k in zip(np.where(r == RUN_B)[0], key[r == RUN_B]) if k.tobytes() in ia]
A = np.array([p[0] for p in pairs]); B = np.array([p[1] for p in pairs])
print(f"run {RUN_A}: {(r==RUN_A).sum()} events in file, run {RUN_B}: {(r==RUN_B).sum()} events in file, matched pairs: {len(pairs)}")

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
sa, sb = x['vt_subrun_number'][A], x['vt_subrun_number'][B]; ea, eb = x['vt_event_number'][A], x['vt_event_number'][B]
axs[0].scatter(sa, sb, s=12); axs[0].set_xlabel(f'subrun in run {RUN_A}'); axs[0].set_ylabel(f'subrun in run {RUN_B}'); axs[0].set_title('subrun mapping of the duplicated pairs', fontsize=10)
for s_a, s_b in sorted(set(zip(sa.tolist(), sb.tolist()))):
    m = (sa == s_a) & (sb == s_b)
    axs[1].plot(ea[m], eb[m], 'o-', ms=4, label=f'subrun {s_a} -> {s_b} ({m.sum()} evts)')
axs[1].set_xlabel(f'event number in run {RUN_A}'); axs[1].set_ylabel(f'event number in run {RUN_B}'); axs[1].legend(fontsize=7); axs[1].set_title('event-number mapping (monotonic within each subrun block)', fontsize=10)
axs[2].hist(ea - eb, bins=30); axs[2].set_xlabel('event number offset (A - B)'); axs[2].set_title('a constant offset per block = same job, re-numbered', fontsize=10)
plt.tight_layout(); plt.show()

## 5. Truth identical, everything downstream different

If the generator event was reused but everything downstream (Geant4 propagation, cosmic overlay, detector simulation, reconstruction) was
run separately, every generator-level quantity should match exactly while Geant4-level truth (e.g. `truth_energyInside`) and everything
reconstructed should differ.

In [ ]:
def compare(name, va, vb):
    same = np.mean([np.array_equal(u, v) for u, v in zip(va, vb)]) if va.dtype == object else np.mean(np.asarray(va, float) == np.asarray(vb, float))
    print(f"   {name:34s} identical in {100*same:5.1f}% of pairs")
print("generator (GENIE) level:")
for f_ in ['vt_mctruth_nu_E','vt_mctruth_lepton_E','vt_mctruth_num_daughter_particles','vt_mctruth_nu_vertex_x','vt_mctruth_nu_vertex_y','vt_mctruth_nu_vertex_z',
           'te_truth_nuTime','vt_mctruth_nu_pdg','vt_mctruth_interaction_type','vt_mctruth_daughters_pdg','vt_mctruth_daughters_E','te_weight_cv','te_weight_spline']:
    compare(f_, x[f_][A], x[f_][B])
print("Geant4-level truth and reconstruction:")
for f_ in ['te_truth_energyInside','vt_reco_vertex_x','vt_reco_vertex_y','vt_reco_vertex_z','vt_reco_asso_showers','vt_reco_asso_tracks','te_match_isFC','te_match_completeness_energy','te_match_energy','tb_numu_cc_flag','tb_nue_score','tb_numu_score']:
    compare(f_, x[f_][A], x[f_][B])

fig, axs = plt.subplots(2, 3, figsize=(14, 7.5))
def xy(ax, va, vb, lab):
    ax.scatter(va, vb, s=10); lo, hi = np.nanmin([va, vb]), np.nanmax([va, vb]); ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
    ax.set_xlabel(f'{lab}, run {RUN_A}'); ax.set_ylabel(f'{lab}, run {RUN_B}')
xy(axs[0,0], x['vt_mctruth_nu_E'][A], x['vt_mctruth_nu_E'][B], r'true $E_\nu$ [GeV]'); axs[0,0].set_title('generator truth: identical', fontsize=10)
xy(axs[0,1], x['vt_mctruth_nu_vertex_z'][A], x['vt_mctruth_nu_vertex_z'][B], 'true vertex z [cm]'); axs[0,1].set_title('generator truth: identical', fontsize=10)
xy(axs[0,2], x['te_weight_cv'][A], x['te_weight_cv'][B], 'GENIE CV tune weight'); axs[0,2].set_title('generator truth: identical', fontsize=10)
xy(axs[1,0], x['te_truth_energyInside'][A], x['te_truth_energyInside'][B], 'Geant4 deposited energy inside [MeV]'); axs[1,0].set_title('Geant4 re-run: differs', fontsize=10)
xy(axs[1,1], x['te_match_completeness_energy'][A], x['te_match_completeness_energy'][B], 'WC match completeness energy [MeV]'); axs[1,1].set_title('reco: differs', fontsize=10)
xy(axs[1,2], x['tb_nue_score'][A], x['tb_nue_score'][B], r'WC $\nu_e$ BDT score'); axs[1,2].set_title('reco: differs', fontsize=10)
plt.tight_layout(); plt.show()

## 5b. One concrete example, self-contained

This cell stands on its own (own imports, no cache): it opens the Run 5 checkout file, pulls two specific run/subrun/event entries, and
prints their truth information side by side. Event-level generator quantities and the full GENIE daughter list are identical; only the
run/subrun/event labels differ.

In [ ]:
import os, uproot, numpy as np
import pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_rows', 60)

EXAMPLE_FILE = "/nevis/riverside/data/leehagaman/ngem/data_files/checkout_MCC9.10_Run4acd5_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist_5.root"
RSE_1 = (25097, 1, 53)
RSE_2 = (25117, 0, 4)

f = uproot.open(EXAMPLE_FILE)
print(f"File: {EXAMPLE_FILE}")
print(f"Events: run/subrun/event {'/'.join(map(str, RSE_1))}  and  {'/'.join(map(str, RSE_2))}\n")
def one_entry(tree, rse, runb, subb, evb, branches):
    cut = f"({runb}=={rse[0]})&({subb}=={rse[1]})&({evb}=={rse[2]})"
    a = tree.arrays(branches, cut=cut, library='ak')
    assert len(a) == 1, f"expected exactly one entry for {rse}, got {len(a)}"
    return a[0]

# --- event-level truth: gLEE vertex_tree (GENIE record) + Wire-Cell T_eval (truth + GENIE weights)
vt = f["singlephotonana/vertex_tree"]; te = f["wcpselection/T_eval"]
vt_ev = ['mctruth_nu_pdg', 'mctruth_ccnc', 'mctruth_mode', 'mctruth_interaction_type', 'mctruth_nu_E', 'mctruth_lepton_E',
         'mctruth_num_daughter_particles', 'mctruth_nu_vertex_x', 'mctruth_nu_vertex_y', 'mctruth_nu_vertex_z',
         'mctruth_num_exiting_protons', 'mctruth_num_exiting_neutrons', 'mctruth_num_exiting_pi0', 'mctruth_num_exiting_pipm']
vt_ev = [b for b in vt_ev if b in vt.keys()]
te_ev = ['truth_nuEnergy', 'truth_nuPdg', 'truth_isCC', 'truth_vtxX', 'truth_vtxY', 'truth_vtxZ', 'truth_nuTime', 'weight_cv', 'weight_spline']
te_ev = [b for b in te_ev if b in te.keys()]

ex_rows = {}
for rse in (RSE_1, RSE_2):
    v = one_entry(vt, rse, 'run_number', 'subrun_number', 'event_number', vt_ev)
    e = one_entry(te, rse, 'run', 'subrun', 'event', te_ev)
    ex_rows['/'.join(map(str, rse))] = {**{'vertex_tree.'+b: v[b] for b in vt_ev}, **{'T_eval.'+b: e[b] for b in te_ev}}
event_table = pd.DataFrame(ex_rows)
event_table['identical'] = event_table.iloc[:, 0].astype(str) == event_table.iloc[:, 1].astype(str)
print(f"=== Event-level truth  [{os.path.basename(EXAMPLE_FILE)}] ===")
print(event_table.to_string())

# --- GENIE daughter particle list from vertex_tree (full record incl. initial state / nuclear remnants)
dbr = ['mctruth_daughters_pdg', 'mctruth_daughters_status_code', 'mctruth_daughters_trackID', 'mctruth_daughters_mother_trackID',
       'mctruth_daughters_E', 'mctruth_daughters_px', 'mctruth_daughters_py', 'mctruth_daughters_pz']
tables = []
for rse in (RSE_1, RSE_2):
    v = one_entry(vt, rse, 'run_number', 'subrun_number', 'event_number', dbr)
    tables.append(pd.DataFrame({b.replace('mctruth_daughters_', ''): np.asarray(v[b]) for b in dbr}))
d1, d2 = tables
print(f"\n=== GENIE daughter particles, {'/'.join(map(str, RSE_1))}  [{os.path.basename(EXAMPLE_FILE)}] ===")
print(d1.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print(f"\n=== GENIE daughter particles, {'/'.join(map(str, RSE_2))}  [{os.path.basename(EXAMPLE_FILE)}] ===")
print(d2.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print(f"\nDaughter tables identical: {d1.equals(d2)}   (event-level: {bool(event_table['identical'].all())})")

## 5c. A cross-file example: Run 1-3 `hist_2_v3` vs Run 4d

Same idea, but the two copies live in **different files and different run periods**: run 12422 (a Run 2 number) in the Run 1-3
$\nu$ overlay `hist_2_v3` file and run 23174 (a Run 4d number) in the Run 4d $\nu$ overlay file. The generator record is identical; the
Geant4-level `truth_energyInside` and everything reconstructed differ.

One generator-stage quantity does differ: `weight_spline`. That is a production difference, not a duplication effect: the Run 4/5 $\nu$ overlay productions store `weight_spline` $= 1$ for every event, whereas the Run 1-3 production carries the real tune-spline weight.

In [ ]:
import os, uproot, numpy as np
import pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_rows', 60)

DATA_DIR_ = "/nevis/riverside/data/leehagaman/ngem/data_files"
PAIR = [  # (file, (run, subrun, event))
    (os.path.join(DATA_DIR_, "checkout_MCC9.10_Run123_v10_04_07_20_BNB_nu_overlay_surprise_reco2_hist_2_v3.root"),  (12422, 86, 4319)),
    (os.path.join(DATA_DIR_, "checkout_MCC9.10_Run4acd5_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist_4d.root"), (23174, 18, 916)),
]

def one_entry(tree, rse, runb, subb, evb, branches):
    a = tree.arrays(branches, cut=f"({runb}=={rse[0]})&({subb}=={rse[1]})&({evb}=={rse[2]})", library='ak')
    assert len(a) == 1, f"expected exactly one entry for {rse}, got {len(a)}"
    return a[0]

vt_ev = ['mctruth_nu_pdg', 'mctruth_ccnc', 'mctruth_mode', 'mctruth_interaction_type', 'mctruth_nu_E', 'mctruth_lepton_E',
         'mctruth_num_daughter_particles', 'mctruth_nu_vertex_x', 'mctruth_nu_vertex_y', 'mctruth_nu_vertex_z',
         'mctruth_num_exiting_protons', 'mctruth_num_exiting_neutrons', 'mctruth_num_exiting_pi0', 'mctruth_num_exiting_pipm']
te_ev = ['truth_nuEnergy', 'truth_nuPdg', 'truth_isCC', 'truth_vtxX', 'truth_vtxY', 'truth_vtxZ', 'truth_nuTime', 'weight_cv', 'weight_spline',
         'truth_energyInside', 'match_isFC', 'match_completeness_energy', 'match_energy']
dbr = ['mctruth_daughters_pdg', 'mctruth_daughters_status_code', 'mctruth_daughters_trackID', 'mctruth_daughters_mother_trackID',
       'mctruth_daughters_E', 'mctruth_daughters_px', 'mctruth_daughters_py', 'mctruth_daughters_pz']

ex_rows, dau_tables, labels = {}, [], []
for path, rse in PAIR:
    f_ = uproot.open(path); vt_ = f_["singlephotonana/vertex_tree"]; te_ = f_["wcpselection/T_eval"]
    label = f"{'Run123 hist_2_v3' if 'Run123' in path else 'Run4d'} | {'/'.join(map(str, rse))}"; labels.append(label)
    v = one_entry(vt_, rse, 'run_number', 'subrun_number', 'event_number', [b for b in vt_ev if b in vt_.keys()])
    e = one_entry(te_, rse, 'run', 'subrun', 'event', [b for b in te_ev if b in te_.keys()])
    ex_rows[label] = {**{'vertex_tree.'+b: v[b] for b in v.fields}, **{'T_eval.'+b: e[b] for b in e.fields}}
    d = one_entry(vt_, rse, 'run_number', 'subrun_number', 'event_number', dbr)
    dau_tables.append(pd.DataFrame({b.replace('mctruth_daughters_', ''): np.asarray(d[b]) for b in dbr}))
    print(f"File: {path}\n   run/subrun/event {'/'.join(map(str, rse))}")

event_table = pd.DataFrame(ex_rows)
event_table['identical'] = event_table.iloc[:, 0].astype(str) == event_table.iloc[:, 1].astype(str)
PRODUCTION_DEPENDENT = ('weight_spline',)      # Run 4/5 productions store weight_spline == 1 for every event; Runs 1-3 carry the real tune-spline weight
DOWNSTREAM = ('energyInside', 'match_')        # Geant4-level truth and reconstruction: expected to differ
def row_class(r_):
    if any(k in r_ for k in DOWNSTREAM): return 'Geant4/reco'
    if any(k in r_ for k in PRODUCTION_DEPENDENT): return 'production-dependent'
    return 'generator'
event_table['level'] = [row_class(r_) for r_ in event_table.index]
gen_rows = event_table.index[event_table['level'] == 'generator']
print("\n=== Event-level truth ===")
print(event_table.to_string())
for lab, d in zip(labels, dau_tables):
    print(f"\n=== GENIE daughter particles  [{lab}] ===")
    print(d.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print(f"\nGenerator-level rows identical: {bool(event_table.loc[gen_rows, 'identical'].all())};  daughter tables identical: {dau_tables[0].equals(dau_tables[1])}")
print("Rows that differ:", ', '.join(event_table.index[~event_table['identical']]))

## 6. What ends up in `all_df.parquet`

The framework keys everything on run/subrun/event, which differ between the copies, so nothing removes them. Repeat the duplicate
search directly inside `all_df.parquet`, per `filetype`, with the four truth floats it carries (energy and vertex; no time, but four exact
float32 matches are still unambiguous).

Because `all_df.parquet` pools all run periods of a filetype, this count also picks up the *cross-period* repeats of section 2 (e.g. `nu_overlay` shows 596 rows in duplicated groups versus 402 summed over the individual files).

In [ ]:
ALL_DF = "/nevis/riverside/data/leehagaman/ngem/intermediate_files/all_df.parquet"
cols = ['filetype', 'run', 'subrun', 'event', 'wc_truth_nuEnergy', 'wc_truth_vtxX', 'wc_truth_vtxY', 'wc_truth_vtxZ']
adf = pl.scan_parquet(ALL_DF).select(cols).collect()
mc_df = adf.filter(pl.col('wc_truth_nuEnergy').is_not_null() & (pl.col('wc_truth_nuEnergy') > 0))
grp = (mc_df.group_by(['filetype', 'wc_truth_nuEnergy', 'wc_truth_vtxX', 'wc_truth_vtxY', 'wc_truth_vtxZ']).len()
             .filter(pl.col('len') > 1))
summary = (mc_df.group_by('filetype').len().rename({'len': 'rows'})
           .join(grp.group_by('filetype').agg(pl.col('len').sum().alias('rows_in_dup_groups'), pl.len().alias('dup_groups')), on='filetype', how='left')
           .fill_null(0).with_columns((100*pl.col('rows_in_dup_groups')/pl.col('rows')).alias('percent')).sort('percent', descending=True))
with pl.Config(tbl_rows=50, tbl_width_chars=200): print(summary)
print("\nrun/subrun/event duplicated within a filetype in all_df:", adf.group_by(['filetype','run','subrun','event']).len().filter(pl.col('len')>1).height)
sub = adf.filter((pl.col('filetype') == 'nu_overlay') & pl.col('run').is_in([RUN_A, RUN_B]))
kA = set(map(tuple, sub.filter(pl.col('run')==RUN_A).select(cols[4:]).to_numpy().tolist())); kB = set(map(tuple, sub.filter(pl.col('run')==RUN_B).select(cols[4:]).to_numpy().tolist()))
print(f"\nRun {RUN_A}/{RUN_B} block: {len(kA & kB)} identical-truth pairs present in both runs inside all_df.parquet (of {len(pairs)} in the checkout file)")

## Conclusion

* **Data and EXT: no duplicated events** within or across files.
* **GENIE MC**: every sample type (nominal, DetVar, dedicated) contains a trickle of generator events that appear twice, always under two
  different run numbers, in subrun-to-subrun blocks with a constant event-number offset, i.e. a generator job's output reused. Generator
  truth and GENIE weights are bit-identical; Geant4, overlay and reconstruction were run independently. The same repeated jobs also show up
  *across* run periods and sample types, so this is a low-rate seed collision in the MCC9 generator production at large. The rate is
  $\sim10^{-4}$ ($\nu$ overlay Runs 4-5: 402 events of 2.65 M; Runs 1-3: none), far below any other uncertainty; both copies propagate into
  `all_df.parquet` and no action is needed there.
* **NuWro Runs 1-3** is the one sample where this is not negligible: 10 % / 4.6 % / 1.5 % of events in the Run 1 / 2 / 3 files are
  duplicates (Runs 4-5: none), and the older MCC9 production (Wire-Cell fake-data files via run/subrun/event, PeLEE ntuples with their own truth) shows the same fractions and the same events, so it is in the NuWro generator sample itself. Histograms are unbiased (independent detsim/reco), but the effective MC statistics of the Run 1-3 NuWro
  fake data are ~5 % lower than the row count, which matters only if one relies on its MC statistical error.